# Loss-rebalance training (Kaggle: T4 ×2 + Internet ON)

**Run All, then walk away (~2–3 hours).** Trains three groups the gate/eval session needs:

1. **stdw** — std-normalized loss, 4 folds × 3 seeds = 12 runs
2. **ptsw** — points-weighted loss, 4 folds × 3 seeds = 12 runs
3. **v1 earlier folds** (through2019/20/21) — baseline, 3 folds × 3 seeds = 9 runs, for the RB out-of-sample test

33 runs total. Nothing here touches the live site — `ARTIFACT_ROOT` stays on v1 until an arm clears the
pre-registered gate (QB weekly Spearman up beyond seed noise; no RB/WR/TE rank regression; MAE, board
hit-rate, calibration all held).

1. Session options → Accelerator **GPU T4 ×2** (not the P100 — sm_60, unsupported), **Internet ON**.
2. Run All.
3. Come back to either a push confirmation or a `loss_rebalance_artifacts.zip` to download and commit
   locally (Kaggle usually can't push with your git creds).

After a session cutoff, just Run All again: completed runs skip themselves and any interrupted run
auto-resumes from its checkpoint. The eval / gate step is a **separate** session — this notebook only trains and saves.

In [ ]:
import os, pathlib, subprocess
root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists()), None)
if root is None:  # fresh Kaggle/Colab session: not inside a clone yet, so make one
    subprocess.run(["git", "clone", "https://github.com/mtsilverstein/Megatron.git"], check=True)
    root = pathlib.Path.cwd() / "Megatron"
os.chdir(root)
print("cwd:", os.getcwd())

!git pull
!pip install -q -e .
!python -m ffmodel.data.pull --seasons 2012 2025 --out data/raw
!python -c "from pathlib import Path; from ffmodel.data.pull import pull_weekly, pull_schedules; from ffmodel.data.features import build_features; s=list(range(2012,2026)); build_features(pull_weekly(s, Path('data/raw')), pull_schedules(s, Path('data/raw'))).to_parquet('data/features_2012_2025.parquet')"
import torch; print('cuda:', torch.cuda.is_available())

In [ ]:
import subprocess

# 33 runs = stdw(4 folds) + ptsw(4 folds) + v1 earlier(3 folds), each x seeds 42/43/44.
# seed 42 is the config default -> NO --seed flag -> run_name dir (stdw/ptsw/v1).
# seeds 43/44 -> --seed suffixes -> {run}_s43 / {run}_s44.
# skip-if-complete makes a re-run after any cutoff resume-only.
groups = {
    "stdw": ["configs/transformer_stdw_through2022.yaml",
             "configs/transformer_stdw_through2023.yaml",
             "configs/transformer_stdw_through2024.yaml",
             "configs/transformer_stdw.yaml"],
    "ptsw": ["configs/transformer_ptsw_through2022.yaml",
             "configs/transformer_ptsw_through2023.yaml",
             "configs/transformer_ptsw_through2024.yaml",
             "configs/transformer_ptsw.yaml"],
    "v1-earlier": ["configs/transformer_v1_through2019.yaml",
                   "configs/transformer_v1_through2020.yaml",
                   "configs/transformer_v1_through2021.yaml"],
}
for seed in (42, 43, 44):
    for group, cfgs in groups.items():
        for cfg in cfgs:
            cmd = ["python", "-m", "ffmodel.model.train", "--config", cfg,
                   "--features-parquet", "data/features_2012_2025.parquet"]
            if seed != 42:
                cmd += ["--seed", str(seed)]
            print()
            print(f"=== seed {seed}  {group}  {cfg} ===", flush=True)
            subprocess.run(cmd, check=True)
print()
print("All 33 runs complete.")

In [ ]:
import subprocess, zipfile
from pathlib import Path

# Commit ONLY the new artifact roots. We deliberately do NOT run the gate here:
# it is a separate controlled eval session (QB weekly Spearman beyond seed noise,
# no RB/WR/TE rank regression, MAE/board/calibration held). Committing these does
# not change the live v1 site -- ARTIFACT_ROOT stays on v1 until an arm passes.
roots = [f"models/transformer/{r}"
         for base in ("stdw", "ptsw")
         for r in (base, f"{base}_s43", f"{base}_s44")]
roots += [f"models/transformer/v1{s}" for s in ("", "_s43", "_s44")]  # earlier folds land here

subprocess.run(["git", "add", *roots], check=False)
subprocess.run(
    ["git", "commit", "-m",
     "model: loss-rebalance artifacts (stdw + ptsw arms) + v1 earlier folds"],
    check=False,
)
pushed = subprocess.run(["git", "push"], check=False).returncode == 0

if pushed:
    print("Pushed loss-rebalance artifacts. Ready for the gate session.")
else:
    zip_path = Path("loss_rebalance_artifacts.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for r in roots:
            for f in Path(r).rglob("*"):
                if f.is_file():
                    zf.write(f, f.relative_to("."))
    for line in [
        f"git push failed -- wrote {zip_path.resolve()} instead.",
        "Download it, then in your local clone:",
        f"  unzip {zip_path.name}",
        "  git add models/transformer/stdw* models/transformer/ptsw* models/transformer/v1*",
        '  git commit -m "model: loss-rebalance artifacts (stdw + ptsw) + v1 earlier folds"',
        "  git push",
    ]:
        print(line)